In [ ]:
#Week 6, Day 3 — June 24, 2026
#Evaluate Random Forest — Export Predictions & Dashboard-Ready Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/HCL_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

In [ ]:
# Load all required datasets
df_c       = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
cluster_df = pd.read_csv(DIRS['processed']+'/cluster_results.csv')
df_c['year']  = df_c['Date'].dt.year
df_c['month'] = df_c['Date'].dt.month

FEATURES_RF = ['SST (°C)', 'pH Level']
TARGET_RF   = 'Species Observed'

# Load trained model
try:
    best_rf = joblib.load(DIRS['processed']+'/rf_best_model.joblib')
    print('Model loaded from disk ✅')
    print(f'  Parameters: {best_rf.get_params()}')
except Exception as e:
    print(f'Model not found on disk ({e}) — retraining...')
    rf_data = df_c[FEATURES_RF+[TARGET_RF]].dropna()
    X = rf_data[FEATURES_RF]; y = rf_data[TARGET_RF]
    X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)
    best_rf = RandomForestRegressor(n_estimators=100,max_depth=3,random_state=42)
    best_rf.fit(X_tr,y_tr)
    print('Model retrained ')

print(f'cluster_df: {cluster_df.shape}')

In [ ]:
#Step 1: Full Model Evaluation Report
# Recreate train/test split (same seed = same split as June 23)
rf_data = df_c[FEATURES_RF+[TARGET_RF]].dropna().reset_index(drop=True)
X_all = rf_data[FEATURES_RF]; y_all = rf_data[TARGET_RF]
X_tr,X_te,y_tr,y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

# Predict on test set and full dataset
yp_test = best_rf.predict(X_te)
yp_all  = best_rf.predict(X_all)

r2_test  = r2_score(y_te,  yp_test)
mae_test = mean_absolute_error(y_te,  yp_test)
r2_all   = r2_score(y_all, yp_all)
mae_all  = mean_absolute_error(y_all, yp_all)

residuals = y_te.values - yp_test

print('RANDOM FOREST MODEL EVALUATION REPORT')
print('='*60)
print()
print('TEST SET PERFORMANCE  (n=100, completely unseen data):')
print(f'  R²   : {r2_test:.4f}  ({r2_test*100:.1f}% of test variance explained)')
print(f'  MAE  : {mae_test:.4f} species (average prediction error)')
print(f'  Residuals: mean={residuals.mean():.4f}, std={residuals.std():.4f}')
print(f'             min={residuals.min():.2f}, max={residuals.max():.2f}')
print()
print('FULL DATASET PERFORMANCE  (n=500, includes training data):')
print(f'  R²   : {r2_all:.4f}')
print(f'  MAE  : {mae_all:.4f} species')
print()
print('FEATURE IMPORTANCES:')
for feat, imp in zip(FEATURES_RF, best_rf.feature_importances_):
    bar = '█' * int(imp * 40)
    print(f'  {feat:<22}: {imp:.4f}  ({imp*100:.2f}%)  {bar}')
print()
print('INTERPRETATION:')
print(f'  Test R²=0.5607 means the model explains 56.1% of species count variance.')
print(f'  MAE=12.11 species means predictions are within ±12 species on average.')
print(f'  SST dominates at 95.31% — each split in every tree starts with SST.')

In [ ]:
#Step 2: Evaluation Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.set_style('whitegrid')

# ── Panel 1: Predicted vs Actual ──
ax1 = axes[0]
ax1.scatter(y_te, yp_test, alpha=0.5, color='steelblue', s=35, edgecolors='none')
lims = [min(float(y_te.min()), yp_test.min())-5,
        max(float(y_te.max()), yp_test.max())+5]
ax1.plot(lims, lims, 'r--', linewidth=2.5, label='Perfect prediction')
ax1.fill_between(lims, [l-mae_test for l in lims],
                 [l+mae_test for l in lims], alpha=0.08, color='orange',
                 label=f'±MAE ({mae_test:.1f} species)')
ax1.set_xlabel('Actual Species Observed', fontsize=11)
ax1.set_ylabel('Predicted Species Observed', fontsize=11)
ax1.set_title(f'Predicted vs Actual\nR² = {r2_test:.4f}  |  MAE = {mae_test:.2f}',
              fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# ── Panel 2: Residuals ──
ax2 = axes[1]
ax2.scatter(yp_test, residuals, alpha=0.5, color='#e74c3c', s=35, edgecolors='none')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=2, label='Zero residual')
ax2.axhline(y=residuals.std(), color='orange', linestyle=':', linewidth=1.5,
            alpha=0.8, label=f'+1σ ({residuals.std():.1f})')
ax2.axhline(y=-residuals.std(), color='orange', linestyle=':', linewidth=1.5,
            alpha=0.8, label=f'-1σ ({-residuals.std():.1f})')
ax2.set_xlabel('Predicted Species', fontsize=11)
ax2.set_ylabel('Residuals (Actual − Predicted)', fontsize=11)
ax2.set_title(f'Residual Plot\nμ={residuals.mean():.2f}, σ={residuals.std():.2f}',
              fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# ── Panel 3: Residual Distribution ──
ax3 = axes[2]
ax3.hist(residuals, bins=20, color='#9b59b6', alpha=0.7, edgecolor='white')
ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
ax3.axvline(x=residuals.mean(), color='orange', linestyle='-',
            linewidth=2, label=f'Mean={residuals.mean():.2f}')
ax3.set_xlabel('Residual Value', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title('Residual Distribution\n(should be approx. normal, centred at 0)',
              fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

plt.suptitle('June 24, 2026 — Random Forest Model Evaluation Report\n'
             'n_estimators=100, max_depth=3, Test R²=0.5607',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week6_Day3_rf_full_evaluation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Full evaluation chart saved ')

In [ ]:
#Step 3: Build Final Export Dataset
# Build the full export dataset:
# original climate data + RF predictions + risk cluster labels + silhouette scores
export_df = df_c[[
    'Date','Location','Latitude','Longitude',
    'SST (°C)','pH Level','Species Observed',
    'Bleaching Severity','Marine Heatwave'
]].dropna(subset=['SST (°C)','pH Level','Species Observed']).reset_index(drop=True)

# Add RF predictions
export_df['RF_Predicted_Species'] = best_rf.predict(export_df[FEATURES_RF]).round(2)
export_df['Prediction_Error']     = (export_df['Species Observed'] - export_df['RF_Predicted_Species']).round(2)
export_df['Abs_Error']            = export_df['Prediction_Error'].abs().round(2)

# Add K-Means risk cluster labels and silhouette scores from Week 5
export_df['Risk_Cluster']         = cluster_df['Risk_Cluster'].values
export_df['Silhouette_Score']     = cluster_df['Silhouette_Score'].values

print(f'FINAL EXPORT DATASET: {export_df.shape[0]} rows × {export_df.shape[1]} cols')
print()
print('Columns in export:')
for col in export_df.columns:
    dtype = export_df[col].dtype
    nulls = export_df[col].isnull().sum()
    print(f'  {col:<30} {str(dtype):<12} nulls={nulls}')
print()
print('Sample predictions:')
print(export_df[['Location','SST (°C)','pH Level','Species Observed',
                  'RF_Predicted_Species','Prediction_Error','Risk_Cluster']].head(5).round(3).to_string())

In [ ]:
#Step 4: Export CSV + PBIX-Compatible Flat File
# ── Export 1: Full dataset with all columns ──
full_path = DIRS['processed']+'/final_predictions_dashboard.csv'
export_df.to_csv(full_path, index=False)
print(f'final_predictions_dashboard.csv  saved ')
print(f'  {len(export_df)} rows × {export_df.shape[1]} cols')
print(f'  All columns including Bleaching Severity, Marine Heatwave')
print()

# ── Export 2: PBIX-compatible flat file (clean, no complex dtypes) ──
pbix_df = export_df[[
    'Date','Location','Latitude','Longitude',
    'SST (°C)','pH Level','Species Observed',
    'RF_Predicted_Species','Prediction_Error','Risk_Cluster'
]].copy()
pbix_df['Date'] = pbix_df['Date'].astype(str)

pbix_path = DIRS['processed']+'/dashboard_ready_data.csv'
pbix_df.to_csv(pbix_path, index=False)
print(f'dashboard_ready_data.csv  saved ')
print(f'  {len(pbix_df)} rows × {pbix_df.shape[1]} cols')
print(f'  Power BI / Tableau compatible (flat, clean dtypes)')
print()

# ── Risk cluster breakdown ──
print('Risk cluster breakdown in export:')
print(export_df['Risk_Cluster'].value_counts().to_string())
print()
print('Prediction accuracy by Risk Cluster:')
for risk in ['Stable','At_Risk','Critical']:
    sub = export_df[export_df['Risk_Cluster']==risk]
    if len(sub)>0:
        print(f'  {risk:<12}: n={len(sub):3d}  '
              f'mean_error={sub["Prediction_Error"].mean():+.2f}  '
              f'mean_abs_error={sub["Abs_Error"].mean():.2f}')

In [ ]:
#Step 5: Final Prediction Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style('whitegrid')
risk_colors = {'Stable':'#27ae60','At_Risk':'#f39c12','Critical':'#e74c3c'}

# ── Panel 1: Predictions colored by Risk Cluster ──
ax1 = axes[0]
for risk, color in risk_colors.items():
    sub = export_df[export_df['Risk_Cluster']==risk]
    ax1.scatter(sub['SST (°C)'], sub['RF_Predicted_Species'],
                c=color, s=25, alpha=0.6, label=f'{risk} (n={len(sub)})',
                edgecolors='none')
ax1.set_xlabel('SST (°C)', fontsize=11)
ax1.set_ylabel('RF Predicted Species', fontsize=11)
ax1.set_title('RF Predictions by Risk Zone\nSST → Predicted Species',
              fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# ── Panel 2: Actual vs Predicted by Risk Cluster ──
ax2 = axes[1]
for risk, color in risk_colors.items():
    sub = export_df[export_df['Risk_Cluster']==risk]
    ax2.scatter(sub['Species Observed'], sub['RF_Predicted_Species'],
                c=color, s=25, alpha=0.6, label=f'{risk} (n={len(sub)})',
                edgecolors='none')
lims2 = [export_df['Species Observed'].min()-5,
         export_df['Species Observed'].max()+5]
ax2.plot(lims2, lims2, 'k--', linewidth=1.5, alpha=0.5, label='y = x')
ax2.set_xlabel('Actual Species Observed', fontsize=11)
ax2.set_ylabel('RF Predicted Species', fontsize=11)
ax2.set_title('Actual vs Predicted by Risk Zone\n(full dataset, n=500)',
              fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle('June 24, 2026 — Predictions Colored by K-Means Risk Zone\n'
             'Random Forest Regressor Output',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week6_Day3_predictions_by_risk.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Predictions by risk zone chart saved ')

In [ ]:
print('ALL JUNE 24 OUTPUTS COMPLETE')
print('='*55)
print()
print('FILES SAVED:')
outputs = [
    ('final_predictions_dashboard.csv', DIRS['processed'], '500 rows, all columns'),
    ('dashboard_ready_data.csv',        DIRS['processed'], 'PBIX-compatible flat file'),
    ('Week6_Day3_rf_full_evaluation.png', DIRS['visualizations'], 'Predicted/Residual/Distribution'),
    ('Week6_Day3_predictions_by_risk.png',DIRS['visualizations'], 'Predictions by risk zone'),
]
for fname, fdir, desc in outputs:
    fpath = fdir+'/'+fname
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)/1024
        print(f'   {fname:<45}  {size:.1f} KB  — {desc}')
    else:
        print(f'   {fname}  NOT FOUND')
print()
